<a href="https://colab.research.google.com/github/25bca118-sudo/ai-chatbot/blob/main/Copy_of_img_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers gradio pillow

In [ ]:
import torch
import gradio as gr
from PIL import Image
from transformers import BlipProcessor, BlipForQuestionAnswering

# Select GPU if available; otherwise use CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Load the pretrained image question-answering model
model_name = "Salesforce/blip-vqa-base"

processor = BlipProcessor.from_pretrained(model_name)
model = BlipForQuestionAnswering.from_pretrained(model_name)
model = model.to(device)
model.eval()

print("Image recognition model loaded successfully!")


Using device: cpu


vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.54GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/788 [00:00<?, ?it/s]

Image recognition model loaded successfully!


In [ ]:
def answer_image_question(image, question):
    """Answer a question about an uploaded image."""

    if image is None:
        return "Please upload an image first."

    if not question or not question.strip():
        question = "What is in this picture?"

    # Convert image to RGB
    image = image.convert("RGB")

    # Prepare the image and question for the model
    inputs = processor(
        images=image,
        text=question.strip(),
        return_tensors="pt"
    ).to(device)

    # Generate an answer
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=40
        )

    answer = processor.decode(
        output[0],
        skip_special_tokens=True
    )

    return answer.strip() or "I could not determine the answer."


def analyze_image(image):
    """Generate an initial description of the uploaded image."""
    return answer_image_question(
        image,
        "What is in this picture?"
    )


def chat(message, history, image):
    """Handle questions in the chatbot."""
    if image is None:
        return "Please upload an image before asking questions."

    return answer_image_question(image, message)


# Build the web interface
with gr.Blocks(title="Image Recognition Chatbot") as demo:
    gr.Markdown(
        """
        # 🖼️ Image Recognition Chatbot
        Upload an image and ask questions about its contents.
        """
    )

    image_input = gr.Image(
        type="pil",
        label="Upload Your Image"
    )

    analyze_button = gr.Button(
        "Analyze Image",
        variant="primary"
    )

    description = gr.Textbox(
        label="Image Recognition Result",
        lines=3
    )

    analyze_button.click(
        fn=analyze_image,
        inputs=image_input,
        outputs=description
    )

    gr.Markdown("## 💬 Chat About Your Image")

    chatbot = gr.ChatInterface(
        fn=chat,
        additional_inputs=[image_input],
        title="Ask Questions",
        description=(
            "Examples: What color is the car? "
            "How many people are there? "
            "What is the person doing?"
        )
    )

demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://cad93f1aba3fd78ca3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
